### 节点缓存

Agent 里最贵的两类节点——**模型调用**和**检索/工具调用**——经常在一次会话里被反复问到同样的输入（重试、重新规划、多用户问同一个问题、断点续跑）。如果每次都真算一遍，钱和延迟都会成倍浪费。LangGraph 为此提供了**节点级缓存**：给节点声明一个 `CachePolicy`，再给图挂一个缓存后端，相同输入的第二次调用就不会再执行节点体。

> 先说清楚它和另外两个「存东西」的东西的区别，避免用错：
>
> | 机制 | 存的是 | 作用范围 | 典型用途 |
> | --- | --- | --- | --- |
> | checkpointer（`InMemorySaver` 等） | **图的状态**（state 快照） | 按 `thread_id` 隔离 | 断点续跑、human-in-the-loop、时间旅行 |
> | **节点缓存**（本节） | **节点的写入**（writes） | 按缓存键共享，**跨 thread** | 复用昂贵的纯计算结果 |
> | store（`BaseStore`） | 长期记忆 / 跨会话数据 | 按 namespace 隔离 | 用户画像、跨会话知识 |
>
> 一句话：checkpointer 回答"上次跑到哪了"，cache 回答"这个结果我算过没有"，store 回答"我还记得什么"。
>
> 本节行为均在 `langgraph==1.2.11` 下实测。


Agent 里最贵的两类节点——**模型调用**和**检索/工具调用**——经常在一次会话里被反复问到同样的输入（重试、重新规划、多用户问同一个问题、断点续跑）。如果每次都真算一遍，钱和延迟都会成倍浪费。LangGraph 为此提供了**节点级缓存**：给节点声明一个 `CachePolicy`，再给图挂一个缓存后端，相同输入的第二次调用就不会再执行节点体。

> 先说清楚它和另外两个「存东西」的东西的区别，避免用错：
>
> | 机制 | 存的是 | 作用范围 | 典型用途 |
> | --- | --- | --- | --- |
> | checkpointer（`InMemorySaver` 等） | **图的状态**（state 快照） | 按 `thread_id` 隔离 | 断点续跑、human-in-the-loop、时间旅行 |
> | **节点缓存**（本节） | **节点的写入**（writes） | 按缓存键共享，**跨 thread** | 复用昂贵的纯计算结果 |
> | store（`BaseStore`） | 长期记忆 / 跨会话数据 | 按 namespace 隔离 | 用户画像、跨会话知识 |
>
> 一句话：checkpointer 回答"上次跑到哪了"，cache 回答"这个结果我算过没有"，store 回答"我还记得什么"。
>
> 本节行为均在 `langgraph==1.2.11` 下实测。


Agent 里最贵的两类节点——**模型调用**和**检索/工具调用**——经常在一次会话里被反复问到同样的输入（重试、重新规划、多用户问同一个问题、断点续跑）。如果每次都真算一遍，钱和延迟都会成倍浪费。LangGraph 为此提供了**节点级缓存**：给节点声明一个 `CachePolicy`，再给图挂一个缓存后端，相同输入的第二次调用就不会再执行节点体。

> 先说清楚它和另外两个「存东西」的东西的区别，避免用错：
>
> | 机制 | 存的是 | 作用范围 | 典型用途 |
> | --- | --- | --- | --- |
> | checkpointer（`InMemorySaver` 等） | **图的状态**（state 快照） | 按 `thread_id` 隔离 | 断点续跑、human-in-the-loop、时间旅行 |
> | **节点缓存**（本节） | **节点的写入**（writes） | 按缓存键共享，**跨 thread** | 复用昂贵的纯计算结果 |
> | store（`BaseStore`） | 长期记忆 / 跨会话数据 | 按 namespace 隔离 | 用户画像、跨会话知识 |
>
> 一句话：checkpointer 回答"上次跑到哪了"，cache 回答"这个结果我算过没有"，store 回答"我还记得什么"。
>
> 本节行为均在 `langgraph==1.2.11` 下实测。


#### 1. 最小用法：两行配置

和 `retry_policy` 一样挂在节点上，但**光声明策略没用**，还必须在 `compile(cache=...)` 时给出缓存后端——两件事都要做。


In [1]:
import time
from operator import add
from typing import Annotated, TypedDict

from langgraph.cache.memory import InMemoryCache
from langgraph.constants import END, START
from langgraph.graph import StateGraph
from langgraph.types import CachePolicy


class CacheState(TypedDict):
    query: str
    logs: Annotated[list[str], add]


exec_count = {"n": 0}   # 统计节点体真正执行的次数


def search_node(state: CacheState) -> CacheState:
    """模拟一次「很贵」的检索 / 模型调用。"""
    exec_count["n"] += 1
    print(f"    [节点体真正执行] 第 {exec_count['n']} 次，query={state['query']}")
    time.sleep(0.2)     # 模拟耗时
    return {"logs": [f"「{state['query']}」的检索结果（执行 #{exec_count['n']}）"]}


cache_builder = StateGraph(CacheState)
# 1) 节点上声明缓存策略
cache_builder.add_node("search_node", search_node, cache_policy=CachePolicy(ttl=10))
cache_builder.add_edge(START, "search_node")
cache_builder.add_edge("search_node", END)

# 2) compile 时提供缓存后端
cache_graph = cache_builder.compile(cache=InMemoryCache())

print("第一次：")
print("  ", cache_graph.invoke({"query": "langgraph 缓存", "logs": []}))
print("第二次（同样的 query）：")
print("  ", cache_graph.invoke({"query": "langgraph 缓存", "logs": []}))
print(f"search_node 真正执行了 {exec_count['n']} 次")


第一次：
    [节点体真正执行] 第 1 次，query=langgraph 缓存


   {'query': 'langgraph 缓存', 'logs': ['「langgraph 缓存」的检索结果（执行 #1）']}
第二次（同样的 query）：
   {'query': 'langgraph 缓存', 'logs': ['「langgraph 缓存」的检索结果（执行 #1）']}
search_node 真正执行了 1 次


注意第二次调用返回的 `logs` 里写的是"执行 #1"——缓存保存并重放的是**节点当时的写入（writes）**，不是重新计算的结果。命中时 `stream_mode="updates"` 还会给这个 chunk 加上一个标记：

```python
{'search_node': {'logs': [...]}, '__metadata__': {'cached': True}}
```

另外，`compile()` 时**不传 `cache` 也不会报错**，`cache_policy` 会被静默忽略（实测执行两次就是真执行两次）。所以"配了没生效"时，先确认 `compile(cache=...)` 有没有写。

#### 2. 缓存键是怎么算出来的

默认的 `key_func` 是 `langgraph._internal._cache.default_cache_key`：先把入参递归"冻结"（`Mapping` 排序成元组、`Sequence` 转元组、numpy/pandas 走 `tobytes()`），再 `pickle.dumps(..., protocol=5)`。最终缓存键是这几段的组合（见 `langgraph/pregel/_algo.py`）：

```python
CacheKey(
    (CACHE_NS_WRITES, f"{模块名}.{函数名}", 节点名),   # 身份：谁算的、在哪个节点上
    xxh3_128_hexdigest(...),                            # 内容：冻结+pickle 后的入参
    ttl,
)
```

**第一个维度是"函数身份"（`module.qualname`），不是函数体**。这一点很关键，后面第 5 节会踩。

#### 3. 默认 key 的坑：state 里任何字段变了都会 miss

因为默认 key 吃的是**节点收到的全部输入**，只要 state 里多了一个和该节点无关的字段（消息历史、计数器、trace id……），键就变了，缓存永远命中不了。这是实践中最常见的"缓存不生效"原因。

解决办法是用 `key_func` 把 key 收窄到"真正影响输出的那几个字段"——`key_func` 收到的是节点输入（通常就是 state），返回 `str` / `bytes` 即可。


In [2]:
def build_graph(cache_policy, cache=None):
    b = StateGraph(CacheState)
    b.add_node("search_node", search_node, cache_policy=cache_policy)
    b.add_edge(START, "search_node")
    b.add_edge("search_node", END)
    return b.compile(cache=cache or InMemoryCache())


exec_count["n"] = 0
print("默认 key：只多带了一个无关字段 logs，也会 miss")
default_key_graph = build_graph(CachePolicy())
default_key_graph.invoke({"query": "默认 key 的坑", "logs": []})
default_key_graph.invoke({"query": "默认 key 的坑", "logs": ["外部塞入的内容"]})
print(f"  真正执行 {exec_count['n']} 次（= 2 次，说明一次都没命中）")

exec_count["n"] = 0
key_builder = StateGraph(CacheState)
# key 只由 query 决定：state 里其它字段怎么变都能命中
key_builder.add_node("search_node", search_node,
                     cache_policy=CachePolicy(key_func=lambda state: state["query"]))
key_builder.add_edge(START, "search_node")
key_builder.add_edge("search_node", END)
key_graph = key_builder.compile(cache=InMemoryCache())

print("自定义 key_func（只按 query）：")
key_graph.invoke({"query": "只按 query 缓存", "logs": []})
key_graph.invoke({"query": "只按 query 缓存", "logs": ["外部塞入的内容"]})
key_graph.invoke({"query": "换个问题", "logs": ["外部塞入的内容"]})
print(f"  真正执行 {exec_count['n']} 次（= 2 次：'只按 query 缓存' 只算了一次）")


默认 key：只多带了一个无关字段 logs，也会 miss
    [节点体真正执行] 第 1 次，query=默认 key 的坑


    [节点体真正执行] 第 2 次，query=默认 key 的坑


  真正执行 2 次（= 2 次，说明一次都没命中）
自定义 key_func（只按 query）：
    [节点体真正执行] 第 1 次，query=只按 query 缓存


    [节点体真正执行] 第 2 次，query=换个问题


  真正执行 2 次（= 2 次：'只按 query 缓存' 只算了一次）


#### 4. 过期与手动失效

- `CachePolicy(ttl=秒数)`：条目存活时间，`None`（默认）表示永不过期。
- `graph.clear_cache(["节点名"])` / `graph.clear_cache()`：手动清缓存（按节点名清或全清）。没传 `cache` 的图调用它会抛 `ValueError`。

生产上建议给缓存设 `ttl`：一是让上游数据更新后能自动生效，二是避免"失败结果被长期缓存"（见第 5 节）。


In [3]:
exec_count["n"] = 0

ttl_builder = StateGraph(CacheState)
ttl_builder.add_node("search_node", search_node, cache_policy=CachePolicy(ttl=1))   # 1 秒后过期
ttl_builder.add_edge(START, "search_node")
ttl_builder.add_edge("search_node", END)
ttl_graph = ttl_builder.compile(cache=InMemoryCache())

ttl_graph.invoke({"query": "带 ttl 的检索", "logs": []})                     # 写入缓存
ttl_graph.invoke({"query": "带 ttl 的检索", "logs": []})                     # 命中
print("立刻再跑 ->", ttl_graph.invoke({"query": "带 ttl 的检索", "logs": []})["logs"])
print("  真正执行次数 =", exec_count["n"])

time.sleep(1.2)                                                             # 等过 ttl
ttl_graph.invoke({"query": "带 ttl 的检索", "logs": []})
print("等 1.2s 后再跑（已过期）->", ttl_graph.invoke({"query": "带 ttl 的检索", "logs": []})["logs"])
print("  真正执行次数 =", exec_count["n"])

print("手动清缓存：")
ttl_graph.clear_cache(["search_node"])          # 也可以 clear_cache() 清掉所有节点
ttl_graph.invoke({"query": "带 ttl 的检索", "logs": []})
print("  clear_cache(['search_node']) 后再跑，真正执行次数 =", exec_count["n"])

try:
    ttl_builder.compile().clear_cache()          # 同样的节点，但 compile 时没传 cache
except ValueError as e:
    print("没传 cache 的图调用 clear_cache ->", type(e).__name__, ":", e)


    [节点体真正执行] 第 1 次，query=带 ttl 的检索


立刻再跑 -> ['「带 ttl 的检索」的检索结果（执行 #1）']
  真正执行次数 = 1


    [节点体真正执行] 第 2 次，query=带 ttl 的检索


等 1.2s 后再跑（已过期）-> ['「带 ttl 的检索」的检索结果（执行 #2）']
  真正执行次数 = 2
手动清缓存：
    [节点体真正执行] 第 3 次，query=带 ttl 的检索


  clear_cache(['search_node']) 后再跑，真正执行次数 = 3
没传 cache 的图调用 clear_cache -> ValueError : No cache is set for this graph. Cannot clear cache.


#### 5. 命中缓存时到底发生了什么

| 现象 | 实测结论 |
| --- | --- |
| 是否执行节点体 | 否，函数一行都不跑 |
| 是否重放 state 写入 | 是，把当初的 writes 原样写回 state |
| 是否消耗 super-step | **是**，缓存只省"节点体的执行"，不省步数（和上一节的 `recursion_limit` 是两回事） |
| `stream_mode="updates"` | 照常吐 chunk，并带 `__metadata__: {'cached': True}` |
| `stream_mode="debug"` | 有 `task` 记录但**没有对应的 `task_result`**（节点没真跑） |
| 是否跨 thread 共享 | **是**，缓存键里没有 `thread_id` |

最后一条要特别小心：缓存实例是全局共享的，`key_func` 里既不能漏掉会影响输出的字段（会串数据），也不能多塞无关字段（会永不命中）。


In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import RetryPolicy

exec_count["n"] = 0
print("--- updates 流：命中缓存的 chunk 带 __metadata__ ---")
for chunk in key_graph.stream({"query": "流式观察", "logs": []}, stream_mode="updates"):
    print("   ", chunk)      # 第一次：没有 __metadata__
for chunk in key_graph.stream({"query": "流式观察", "logs": []}, stream_mode="updates"):
    print("   ", chunk)      # 第二次：{'search_node': {...}, '__metadata__': {'cached': True}}

print("--- debug 流：命中缓存的 task 没有 task_result（沿用上面已缓存的 query）---")
for record in key_graph.stream({"query": "流式观察", "logs": []}, stream_mode="debug"):
    payload = record["payload"]
    print(f"    step={record['step']} {record['type']:<14} node={payload.get('name', '')} "
          f"error={payload.get('error')}")

print("--- 缓存跨 thread 共享（键里没有 thread_id）---")
exec_count["n"] = 0
shared_cache = InMemoryCache()
shared_builder = StateGraph(CacheState)
shared_builder.add_node("search_node", search_node,
                        cache_policy=CachePolicy(key_func=lambda s: s["query"]))
shared_builder.add_edge(START, "search_node")
shared_builder.add_edge("search_node", END)
shared_graph = shared_builder.compile(cache=shared_cache, checkpointer=InMemorySaver())

shared_graph.invoke({"query": "跨 thread 的问题", "logs": []}, {"configurable": {"thread_id": "user-A"}})
print("  user-A 问完，真正执行次数 =", exec_count["n"])
shared_graph.invoke({"query": "跨 thread 的问题", "logs": []}, {"configurable": {"thread_id": "user-B"}})
print("  user-B 问同一个问题（新的 thread），真正执行次数 =", exec_count["n"],
      "  <- 复用了 user-A 写入的缓存")


--- updates 流：命中缓存的 chunk 带 __metadata__ ---
    [节点体真正执行] 第 1 次，query=流式观察


    {'search_node': {'logs': ['「流式观察」的检索结果（执行 #1）']}}
    {'search_node': {'logs': ['「流式观察」的检索结果（执行 #1）']}, '__metadata__': {'cached': True}}
--- debug 流：命中缓存的 task 没有 task_result（沿用上面已缓存的 query）---
    step=1 task           node=search_node error=None
--- 缓存跨 thread 共享（键里没有 thread_id）---


    [节点体真正执行] 第 1 次，query=跨 thread 的问题


  user-A 问完，真正执行次数 = 1
  user-B 问同一个问题（新的 thread），真正执行次数 = 1   <- 复用了 user-A 写入的缓存


#### 6. 三个必须知道的陷阱

1. **改了节点实现但函数名不变 → 旧缓存继续命中**。缓存键用的是 `模块名.函数名`，不含函数体。改了 `search_node` 的实现（比如接了新的检索逻辑）却没清缓存，用户会一直拿到旧结果；反过来改个函数名（`search_node_v2`）就会被当成新节点自动 miss。开发期最稳的做法是：**改实现就 `clear_cache()`**。

2. **失败的结果也会被缓存**。节点抛异常时，这一轮同样会往缓存里写东西（空写入集）。于是同输入的第二次调用**不会抛异常、也不会执行节点体**，而是以"什么都没写"结束——实测下游节点也不会被触发，图悄悄返回了一个残缺结果。这与上一节"失败也要靠 `retry_policy` 扛过去"是配套的：**给可能失败的节点配重试，让它重试到成功再进缓存**，或者干脆不给它开缓存、又或者用短 `ttl` 限制脏数据存活。

3. **`key` 要能 pickle**。默认 `key_func` 走 `pickle`，state 里放了 lambda、打开的文件句柄、数据库连接这类不可序列化对象时，会直接抛 `PicklingError`；这类节点要么自定义 `key_func` 绕开，要么把这类对象移出 state。


In [5]:
print("陷阱 1：改了实现但函数名不变 -> 旧缓存继续命中")
version = {"v": 1}


def versioned_node(state: CacheState) -> CacheState:
    return {"logs": [f"v{version['v']} 的检索结果"]}


trap_builder = StateGraph(CacheState)
trap_builder.add_node("search_node", versioned_node, cache_policy=CachePolicy())
trap_builder.add_edge(START, "search_node")
trap_builder.add_edge("search_node", END)
trap_graph = trap_builder.compile(cache=InMemoryCache())

print("  v1 首次:", trap_graph.invoke({"query": "陷阱 1", "logs": []})["logs"])
version["v"] = 2                                     # 模拟「改了实现」
print("  改成 v2 后再问:", trap_graph.invoke({"query": "陷阱 1", "logs": []})["logs"], " <- 仍是 v1 的旧结果")
trap_graph.clear_cache(["search_node"])
print("  clear_cache 之后:", trap_graph.invoke({"query": "陷阱 1", "logs": []})["logs"])

print("陷阱 2：失败的结果也会被缓存")
fail_count = {"n": 0}


def flaky_node(state: CacheState) -> CacheState:
    fail_count["n"] += 1
    if fail_count["n"] == 1:
        raise ValueError("第一次失败（偶发抖动）")
    return {"logs": ["第二次成功"]}


fail_builder = StateGraph(CacheState)
fail_builder.add_node("flaky_node", flaky_node, cache_policy=CachePolicy())
fail_builder.add_node("after", lambda state: {"logs": ["after 本该执行"]})
fail_builder.add_edge(START, "flaky_node")
fail_builder.add_edge("flaky_node", "after")
fail_builder.add_edge("after", END)
fail_graph = fail_builder.compile(cache=InMemoryCache())

try:
    fail_graph.invoke({"query": "陷阱 2", "logs": []})
except ValueError as e:
    print("  第一次:", type(e).__name__, e)
print("  第二次:", fail_graph.invoke({"query": "陷阱 2", "logs": []}),
      " <- 没再抛错、也没执行节点，'after' 也没跑：失败被当成空结果缓存了")

print("  修法：配上 retry_policy，让它重试到成功（成功结果才进缓存）")
retry_count = {"n": 0}


def retry_flaky_node(state: CacheState) -> CacheState:
    retry_count["n"] += 1
    if retry_count["n"] == 1:
        raise ValueError("第一次失败（偶发抖动）")
    return {"logs": [f"第 {retry_count['n']} 次成功"]}


retry_builder = StateGraph(CacheState)
retry_builder.add_node(
    "flaky_node", retry_flaky_node,
    cache_policy=CachePolicy(),
    retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.001, retry_on=ValueError),
)
retry_builder.add_node("after", lambda state: {"logs": ["after 执行了"]})
retry_builder.add_edge(START, "flaky_node")
retry_builder.add_edge("flaky_node", "after")
retry_builder.add_edge("after", END)
retry_graph = retry_builder.compile(cache=InMemoryCache())

print("  首次:", retry_graph.invoke({"query": "陷阱 2 修法", "logs": []}))
print("  再来一次:", retry_graph.invoke({"query": "陷阱 2 修法", "logs": []}), " 执行次数 =", retry_count["n"])

print("陷阱 3：state 里放不可 pickle 的对象")
class WeirdState(TypedDict):
    payload: object


def echo_node(state: WeirdState) -> WeirdState:
    return {}


pickle_builder = StateGraph(WeirdState)
pickle_builder.add_node("echo_node", echo_node, cache_policy=CachePolicy())
pickle_builder.add_edge(START, "echo_node")
pickle_builder.add_edge("echo_node", END)
pickle_graph = pickle_builder.compile(cache=InMemoryCache())
try:
    pickle_graph.invoke({"payload": lambda x: x})       # lambda 没法 pickle
except Exception as e:
    print(" ", type(e).__name__, ":", str(e)[:90])


陷阱 1：改了实现但函数名不变 -> 旧缓存继续命中
  v1 首次: ['v1 的检索结果']
  改成 v2 后再问: ['v1 的检索结果']  <- 仍是 v1 的旧结果
  clear_cache 之后: ['v2 的检索结果']
陷阱 2：失败的结果也会被缓存
  第一次: ValueError 第一次失败（偶发抖动）
  第二次: {'query': '陷阱 2', 'logs': []}  <- 没再抛错、也没执行节点，'after' 也没跑：失败被当成空结果缓存了
  修法：配上 retry_policy，让它重试到成功（成功结果才进缓存）
  首次: {'query': '陷阱 2 修法', 'logs': ['第 2 次成功', 'after 执行了']}
  再来一次: {'query': '陷阱 2 修法', 'logs': ['第 2 次成功', 'after 执行了']}  执行次数 = 2
陷阱 3：state 里放不可 pickle 的对象
  PicklingError : Can't pickle <function <lambda> at 0x10dabf600>: attribute lookup <lambda> on __main__ fai


#### 7. 和 checkpointer / retry 的关系

- **与 checkpointer 同用**：同一个 `thread_id` 的第二次 `invoke` 是"续跑"——它会接着上次的 state 往下走，节点输入自然和第一次不同，默认 key 必然 miss。想让续跑也命中缓存，就得用自定义 `key_func` 只取真正影响输出的字段。
- **与 retry 同用**：重试到成功的那一次结果会被正常缓存（失败的那些尝试不会）。所以"重试 + 缓存"是安全组合，而"只缓存不重试"会踩上面第 2 个陷阱。

#### 8. Agent 场景：缓存工具 / 模型节点

`ToolNode`、`create_react_agent` 里的工具节点同样可以挂 `cache_policy`。命中时 `ToolMessage`（乃至 `AIMessage`）会被原样重放——注意这意味着**模型也会看到上次那份工具结果**，所以缓存键必须覆盖"会改变工具结果的所有输入"（城市、时间窗、用户权限……）。


In [6]:
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState
from langgraph.prebuilt import ToolNode

weather_calls = {"n": 0}


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """查询指定城市的天气。

    Args:
        city: 城市名称，例如「北京」。

    Returns:
        该城市的天气描述字符串。
    """
    weather_calls["n"] += 1
    return f"{city}：晴，{20 + weather_calls['n']}°C（这是第 {weather_calls['n']} 次真实调用）"


agent_cache_builder = StateGraph(MessagesState)
agent_cache_builder.add_node("tools", ToolNode([get_weather]), cache_policy=CachePolicy())
agent_cache_builder.add_edge(START, "tools")
agent_cache_builder.add_edge("tools", END)
agent_cache_graph = agent_cache_builder.compile(cache=InMemoryCache())

ai_message = AIMessage(
    content="",
    tool_calls=[{"name": "get_weather", "args": {"city": "北京"}, "id": "call_1", "type": "tool_call"}],
)
first = agent_cache_graph.invoke({"messages": [ai_message]})
second = agent_cache_graph.invoke({"messages": [ai_message]})
print("工具真实调用次数 =", weather_calls["n"])
print("第一次 ToolMessage =", first["messages"][-1].content)
print("第二次 ToolMessage =", second["messages"][-1].content, " <- 原样重放")

# 全图默认缓存策略：给每个节点都挂上（error_handler 节点不在此列）
print("--- set_node_defaults(cache_policy=...) 给全图兜底 ---")
exec_count["n"] = 0
defaults_builder = StateGraph(CacheState)
defaults_builder.set_node_defaults(cache_policy=CachePolicy(key_func=lambda s: s["query"]))
defaults_builder.add_node("search_node", search_node)
defaults_builder.add_edge(START, "search_node")
defaults_builder.add_edge("search_node", END)
defaults_graph = defaults_builder.compile(cache=InMemoryCache())
defaults_graph.invoke({"query": "全图默认策略", "logs": []})
defaults_graph.invoke({"query": "全图默认策略", "logs": ["无关字段"]})
print("  真正执行次数 =", exec_count["n"], "（节点上没写 cache_policy，也照样命中）")


工具真实调用次数 = 1
第一次 ToolMessage = 北京：晴，21°C（这是第 1 次真实调用）
第二次 ToolMessage = 北京：晴，21°C（这是第 1 次真实调用）  <- 原样重放
--- set_node_defaults(cache_policy=...) 给全图兜底 ---
    [节点体真正执行] 第 1 次，query=全图默认策略


  真正执行次数 = 1 （节点上没写 cache_policy，也照样命中）


#### 9. 缓存后端怎么选

| 后端 | 导入路径 | 特点 |
| --- | --- | --- |
| 内存 | `langgraph.cache.memory.InMemoryCache` | 零依赖、进程内，**重启即失效、多进程不共享**（本仓库用 `langgraph-checkpoint` 系列之外的内置实现） |
| Redis | `langgraph.cache.redis.RedisCache` | 跨进程/跨机器共享，需要额外装 `redis` 并起服务 |
| SQLite | 需另装 `langgraph-checkpoint-sqlite` | 单文件持久化，适合本地/单机部署（本仓库未安装） |
| 自定义 | 实现 `langgraph.cache.base.BaseCache` | 只有 6 个方法：`get/set/aget/aset/clear/aclear` |

本节所有例子都用 `InMemoryCache`。换成 Redis 只需改一行 `compile(cache=RedisCache(...))`，节点配置完全不用动——这也是把缓存抽象成"策略 + 后端"两层的好处。

```python
from langgraph.cache.base import BaseCache

print("自定义后端只需实现：", sorted(BaseCache.__abstractmethods__))
```


In [7]:
from langgraph.cache.base import BaseCache

print("自定义缓存后端只需实现：", sorted(BaseCache.__abstractmethods__))


自定义缓存后端只需实现： ['aclear', 'aget', 'aset', 'clear', 'get', 'set']


#### 实践清单

1. **只给"贵且幂等"的节点开缓存**：模型调用、检索、embedding、汇率/天气查询；有副作用的节点（下单、发消息、扣款、写库）绝对不要缓存。
2. **一定要显式写 `key_func`**，把键收窄到"真正影响输出的字段"，否则 state 一变就 miss，缓存等于白配；反过来也别把 `user_id`、时间窗这类影响输出的字段漏掉，会串数据。
3. **给 `ttl`**，让上游数据更新能自动生效，也避免脏结果长期驻留。
4. **给可能失败的节点配 `retry_policy`**，让失败在进缓存前就被重试掉；否则偶发失败会被缓存成"空结果"，后续请求悄悄拿到残缺 state。
5. **改节点实现后记得 `clear_cache()`**：缓存键只认 `模块名.函数名`，函数体变了缓存不会自动失效。
6. **别把不可 pickle 的对象放进 state**（lambda、连接、文件句柄），否则默认 `key_func` 直接抛 `PicklingError`。
7. **缓存是跨 thread 共享的**，多租户场景要么把租户维度写进 `key_func`，要么给每个租户独立的缓存实例。
8. **命中不等于省步数**：缓存只跳过节点体，super-step 照样消耗，所以它不能替代 `recursion_limit` / 业务步数预算（见 `05-AgentLoop.ipynb`）。
9. **多进程部署要用共享后端**（Redis / SQLite / 自定义），`InMemoryCache` 只在单进程内有效；用 `stream_mode="updates"` 里的 `__metadata__.cached` 可以在灰度时统计命中率。
